Resumen del notebook: Se carga la copia local de RouterBench, comprobamos y limpiamos los datos, seleccionamos tres categorías de consultas y separamos en train-test. Con los datos de estimación calculamos los costes medios y las tasas de acierto de cada modelo para cada categoría

In [16]:
from pathlib import Path
import pandas as pd

data_dir = Path("routerbench")
data_file = data_dir / "routerbench_0shot.pkl"

df = pd.read_pickle(data_file)

print("Rows and columns:", df.shape)
display(df.head())

Rows and columns: (36497, 37)


,sample_id,prompt,eval_name,WizardLM/WizardLM-13B-V1.2,claude-instant-v1,claude-v1,claude-v2,gpt-3.5-turbo-1106,gpt-4-1106-preview,meta/code-llama-instruct-34b-chat,...,claude-v1|total_cost,claude-v2|total_cost,gpt-4-1106-preview|total_cost,meta/llama-2-70b-chat|total_cost,mistralai/mixtral-8x7b-chat|total_cost,zero-one-ai/Yi-34B-Chat|total_cost,WizardLM/WizardLM-13B-V1.2|total_cost,meta/code-llama-instruct-34b-chat|total_cost,mistralai/mistral-7b-chat|total_cost,oracle_model_to_route_to
0,Chinese_character_riddles.dev.0,['猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)...,Chinese_character_riddles,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.004904,0.006488,0.00973,0.000266,0.000166,0.000250,0.000103,0.000236,0.000062,no_model_correct
1,Chinese_character_riddles.dev.1,['猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)...,Chinese_character_riddles,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.003096,0.004704,0.01077,0.000257,0.000227,0.000226,0.000073,0.000128,0.000050,no_model_correct
2,Chinese_character_riddles.dev.10,['猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)...,Chinese_character_riddles,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.002480,0.003728,0.01105,0.000232,0.000215,0.000215,0.000138,0.000150,0.000051,zero-one-ai/Yi-34B-Chat
3,Chinese_character_riddles.dev.11,['猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)...,Chinese_character_riddles,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.003088,0.004960,0.00908,0.000287,0.000117,0.000318,0.000104,0.000315,0.000045,zero-one-ai/Yi-34B-Chat
4,Chinese_character_riddles.dev.12,['猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)...,Chinese_character_riddles,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.002272,0.004912,0.01322,0.000254,0.000097,0.000210,0.000112,0.000190,0.000039,no_model_correct


In [17]:
# Inspect column names
print(df.columns.tolist())

# Inspect data types and non-null counts
df.info()

['sample_id', 'prompt', 'eval_name', 'WizardLM/WizardLM-13B-V1.2', 'claude-instant-v1', 'claude-v1', 'claude-v2', 'gpt-3.5-turbo-1106', 'gpt-4-1106-preview', 'meta/code-llama-instruct-34b-chat', 'meta/llama-2-70b-chat', 'mistralai/mistral-7b-chat', 'mistralai/mixtral-8x7b-chat', 'zero-one-ai/Yi-34B-Chat', 'gpt-3.5-turbo-1106|model_response', 'claude-instant-v1|model_response', 'claude-v1|model_response', 'claude-v2|model_response', 'gpt-4-1106-preview|model_response', 'meta/llama-2-70b-chat|model_response', 'mistralai/mixtral-8x7b-chat|model_response', 'zero-one-ai/Yi-34B-Chat|model_response', 'WizardLM/WizardLM-13B-V1.2|model_response', 'meta/code-llama-instruct-34b-chat|model_response', 'mistralai/mistral-7b-chat|model_response', 'gpt-3.5-turbo-1106|total_cost', 'claude-instant-v1|total_cost', 'claude-v1|total_cost', 'claude-v2|total_cost', 'gpt-4-1106-preview|total_cost', 'meta/llama-2-70b-chat|total_cost', 'mistralai/mixtral-8x7b-chat|total_cost', 'zero-one-ai/Yi-34B-Chat|total_cos

In [18]:
# Count samples in each benchmark
display(df["eval_name"].value_counts())

eval_name
hellaswag                    10042
grade-school-math             7450
mmlu-professional-law         1534
arc-challenge                 1470
winogrande                    1267
                             ...  
chinese-remainder-theorem       15
chinese_chu_ci                  15
chinese_idioms                  15
chinese_poem                    15
test-match                       3
Name: count, Length: 86, dtype: int64

In [19]:
# Inspect evaluation values for one model
model_name = "gpt-3.5-turbo-1106"

for value in df[model_name].head(5):
    print(type(value).__name__, repr(value))

# Inspect the corresponding costs
display(df[[model_name, f"{model_name}|total_cost"]].head())

float 0.0
float 0.0
float 0.0
float 0.0
float 0.0


,gpt-3.5-turbo-1106,gpt-3.5-turbo-1106|total_cost
0,0.0,0.000527
1,0.0,0.000205
2,0.0,0.000368
3,0.0,0.000243
4,0.0,0.000227


In [20]:
# Identify models from their cost columns
models = [
    column.removesuffix("|total_cost")
    for column in df.columns
    if column.endswith("|total_cost")
]

print("Number of models:", len(models))

for model in models:
    print(model)

Number of models: 11
gpt-3.5-turbo-1106
claude-instant-v1
claude-v1
claude-v2
gpt-4-1106-preview
meta/llama-2-70b-chat
mistralai/mixtral-8x7b-chat
zero-one-ai/Yi-34B-Chat
WizardLM/WizardLM-13B-V1.2
meta/code-llama-instruct-34b-chat
mistralai/mistral-7b-chat


In [21]:
# Check evaluation scores and costs for every model
validation_rows = []

for model in models:
    raw_scores = df[model]
    raw_costs = df[f"{model}|total_cost"]

    scores = pd.to_numeric(raw_scores, errors="coerce")
    costs = pd.to_numeric(raw_costs, errors="coerce")

    validation_rows.append({
        "model": model,
        "missing_scores": raw_scores.isna().sum(),
        "non_numeric_scores": (
            raw_scores.notna() & scores.isna()
        ).sum(),
        "non_binary_scores": (
            scores.notna() & ~scores.isin([0, 1])
        ).sum(),
        "missing_costs": raw_costs.isna().sum(),
        "non_numeric_costs": (
            raw_costs.notna() & costs.isna()
        ).sum(),
        "negative_costs": (costs < 0).sum(),
        "infinite_values": (
            scores.isin([float("inf"), float("-inf")]).sum()
            + costs.isin([float("inf"), float("-inf")]).sum()
        )
    })

validation_summary = pd.DataFrame(validation_rows)
display(validation_summary)

,model,missing_scores,non_numeric_scores,non_binary_scores,missing_costs,non_numeric_costs,negative_costs,infinite_values
0,gpt-3.5-turbo-1106,0,0,7737,0,0,0,0
1,claude-instant-v1,0,0,7669,0,0,0,0
2,claude-v1,0,0,7637,0,0,0,0
3,claude-v2,0,0,7771,0,0,0,0
4,gpt-4-1106-preview,0,0,7719,0,0,0,0
5,meta/llama-2-70b-chat,0,0,7782,0,0,0,0
6,mistralai/mixtral-8x7b-chat,0,0,7683,0,0,0,0
7,zero-one-ai/Yi-34B-Chat,0,0,7705,0,0,0,0
8,WizardLM/WizardLM-13B-V1.2,0,0,7710,0,0,0,0
9,meta/code-llama-instruct-34b-chat,0,0,7758,0,0,0,0


In [22]:
# Check missing identifiers and repeated sample IDs
print("Missing sample IDs:", df["sample_id"].isna().sum())
print("Missing benchmark names:", df["eval_name"].isna().sum())
print("Repeated sample IDs:", df["sample_id"].duplicated().sum())

# Display every benchmark and its sample count
benchmark_counts = (
    df["eval_name"]
    .value_counts(dropna=False)
    .rename_axis("benchmark")
    .reset_index(name="sample_count")
)

print(benchmark_counts.to_string(index=False))

Missing sample IDs: 0
Missing benchmark names: 0
Repeated sample IDs: 0
                               benchmark  sample_count
                               hellaswag         10042
                       grade-school-math          7450
                   mmlu-professional-law          1534
                           arc-challenge          1470
                              winogrande          1267
                    mmlu-moral-scenarios           895
                      mmlu-miscellaneous           783
            mmlu-professional-psychology           612
             mmlu-high-school-psychology           545
                                    mbpp           427
                          chinese_zodiac           394
         mmlu-high-school-macroeconomics           390
             mmlu-elementary-mathematics           378
                       consensus_summary           362
                     mmlu-moral-disputes           346
                         mmlu-prehistory        

In [23]:
# Locate non-binary scores by benchmark and model
non_binary_rows = []

for model in models:
    scores = pd.to_numeric(df[model], errors="coerce")
    non_binary_mask = scores.notna() & ~scores.isin([0, 1])

    for benchmark, group in df.loc[non_binary_mask].groupby("eval_name"):
        values = scores.loc[group.index]

        non_binary_rows.append({
            "benchmark": benchmark,
            "model": model,
            "non_binary_count": len(values),
            "min_score": values.min(),
            "max_score": values.max(),
            "example_values": values.unique()[:5].tolist()
        })

non_binary_summary = pd.DataFrame(non_binary_rows)

if non_binary_summary.empty:
    print("All non-missing numeric scores are binary.")
else:
    print(non_binary_summary.to_string(index=False))

        benchmark                             model  non_binary_count  min_score  max_score            example_values
   chinese_idioms                gpt-3.5-turbo-1106                10       0.25       0.75              [0.25, 0.75]
consensus_summary                gpt-3.5-turbo-1106               272       0.25       0.75         [0.75, 0.5, 0.25]
grade-school-math                gpt-3.5-turbo-1106              7404       0.25       0.75         [0.5, 0.75, 0.25]
          mtbench                gpt-3.5-turbo-1106                26       0.10       0.90 [0.7, 0.9, 0.5, 0.8, 0.1]
     mtbench-math                gpt-3.5-turbo-1106                12       0.10       0.50      [0.1, 0.2, 0.5, 0.3]
mtbench-reference                gpt-3.5-turbo-1106                13       0.10       0.80 [0.3, 0.2, 0.1, 0.8, 0.7]
   chinese_idioms                 claude-instant-v1                 7       0.25       0.75              [0.25, 0.75]
consensus_summary                 claude-instant-v1     

In [24]:
# Define candidate categories using benchmarks with binary scores
category_mapping = {
    "mmlu-elementary-mathematics": "mathematics",
    "mmlu-high-school-mathematics": "mathematics",
    "mmlu-college-mathematics": "mathematics",
    "mmlu-abstract-algebra": "mathematics",
    "mbpp": "programming",
    "arc-challenge": "science"
}

# Create a separate working dataset
selected_df = df[
    df["eval_name"].isin(category_mapping)
].copy()

selected_df["category"] = selected_df["eval_name"].map(
    category_mapping
)

# Convert scores and verify that every selected value is binary
selected_df[models] = selected_df[models].apply(
    pd.to_numeric, errors="raise"
)

assert selected_df[models].isin([0, 1]).all().all(), (
    "Some selected scores are missing or non-binary."
)

# Summarize the selected data
category_summary = (
    selected_df.groupby("category")
    .agg(
        sample_count=("sample_id", "size"),
        benchmark_count=("eval_name", "nunique")
    )
)

display(category_summary)

,sample_count,benchmark_count
category,,
mathematics,848,4
programming,427,1
science,1470,1


In [25]:
# Identify repeated prompts after removing surrounding whitespace
selected_df["prompt_key"] = (
    selected_df["prompt"].astype(str).str.strip()
)

assert selected_df["prompt_key"].ne("").all(), "Empty prompts found."

benchmark_overlap = (
    selected_df.groupby("prompt_key")["eval_name"].nunique()
)

assert benchmark_overlap.le(1).all(), (
    "Some prompts appear in multiple benchmarks. Review them first."
)

duplicate_count = selected_df["prompt_key"].duplicated().sum()
print("Repeated prompts removed:", duplicate_count)

clean_df = selected_df.drop_duplicates(
    subset="prompt_key", keep="first"
).copy()

print("Remaining samples:", len(clean_df))

Repeated prompts removed: 2
Remaining samples: 2743


In [26]:
# Split each benchmark into estimation (80%) and evaluation (20%) samples
random_seed = 42
estimation_parts = []
evaluation_parts = []

for benchmark, group in clean_df.groupby("eval_name"):
    assert len(group) >= 2, f"Not enough samples in {benchmark}."

    estimation_size = max(
        1, min(len(group) - 1, int(0.8 * len(group)))
    )

    estimation_part = group.sample(
        n=estimation_size,
        random_state=random_seed
    )

    evaluation_part = group.drop(estimation_part.index)

    estimation_parts.append(estimation_part)
    evaluation_parts.append(evaluation_part)

estimation_df = pd.concat(estimation_parts).copy()
evaluation_df = pd.concat(evaluation_parts).copy()

# Verify that no prompt appears in both sets
assert set(estimation_df["prompt_key"]).isdisjoint(
    set(evaluation_df["prompt_key"])
)

split_summary = pd.concat(
    [
        estimation_df.groupby("category").size().rename("estimation"),
        evaluation_df.groupby("category").size().rename("evaluation")
    ],
    axis=1
)

display(split_summary)

,estimation,evaluation
category,,
mathematics,678,170
programming,340,85
science,1176,294


In [27]:
# Estimate model performance and cost by category
#ONLY WITH THE TRAINING PART OG THE DATA
parameter_rows = []

for category, group in estimation_df.groupby("category"):
    for model in models:
        scores = pd.to_numeric(group[model], errors="raise")
        costs = pd.to_numeric(
            group[f"{model}|total_cost"], errors="raise"
        )

        assert scores.isin([0, 1]).all()
        assert costs.notna().all()
        assert costs.ge(0).all()
        assert ~costs.isin([float("inf"), float("-inf")]).any()

        parameter_rows.append({
            "category": category,
            "model": model,
            "mean_cost": costs.mean(),
            "accuracy": scores.mean(),
            "sample_count": len(group)
        })

all_parameters = pd.DataFrame(parameter_rows)

print("Estimated accuracy by category:")
display(
    all_parameters.pivot(
        index="model", columns="category", values="accuracy"
    ).round(3)
)

print("Estimated mean cost per query by category:")
display(
    all_parameters.pivot(
        index="model", columns="category", values="mean_cost"
    ).round(6)
)

Estimated accuracy by category:


category,mathematics,programming,science
model,,,
WizardLM/WizardLM-13B-V1.2,0.251,0.365,0.618
claude-instant-v1,0.404,0.597,0.800
claude-v1,0.389,0.591,0.869
claude-v2,0.403,0.638,0.866
gpt-3.5-turbo-1106,0.392,0.641,0.831
gpt-4-1106-preview,0.580,0.682,0.961
meta/code-llama-instruct-34b-chat,0.001,0.524,0.355
meta/llama-2-70b-chat,0.009,0.309,0.740
mistralai/mistral-7b-chat,0.168,0.326,0.385


Estimated mean cost per query by category:


category,mathematics,programming,science
model,,,
WizardLM/WizardLM-13B-V1.2,0.000031,0.000082,0.000027
claude-instant-v1,0.000085,0.000485,0.000073
claude-v1,0.000845,0.004351,0.000730
claude-v2,0.000845,0.005893,0.000730
gpt-3.5-turbo-1106,0.000105,0.000338,0.000091
gpt-4-1106-preview,0.001061,0.009426,0.000923
meta/code-llama-instruct-34b-chat,0.000081,0.000158,0.000070
meta/llama-2-70b-chat,0.000094,0.000301,0.000081
mistralai/mistral-7b-chat,0.000021,0.000050,0.000018


In [28]:
# Define the initial three-model case study
selected_models = [
    "gpt-3.5-turbo-1106",
    "gpt-4-1106-preview",
    "mistralai/mixtral-8x7b-chat"
]

assert set(selected_models).issubset(models)

model_parameters = all_parameters[
    all_parameters["model"].isin(selected_models)
].copy()

model_parameters = model_parameters.sort_values(
    ["category", "model"]
).reset_index(drop=True)

assert len(model_parameters) == 9
assert not model_parameters.duplicated(
    subset=["category", "model"]
).any()

display(model_parameters)

,category,model,mean_cost,accuracy,sample_count
0,mathematics,gpt-3.5-turbo-1106,0.000105,0.392330,678
1,mathematics,gpt-4-1106-preview,0.001061,0.579646,678
2,mathematics,mistralai/mixtral-8x7b-chat,0.000063,0.221239,678
3,programming,gpt-3.5-turbo-1106,0.000338,0.641176,340
4,programming,gpt-4-1106-preview,0.009426,0.682353,340
5,programming,mistralai/mixtral-8x7b-chat,0.000176,0.535294,340
6,science,gpt-3.5-turbo-1106,0.000091,0.830782,1176
7,science,gpt-4-1106-preview,0.000923,0.960884,1176
8,science,mistralai/mixtral-8x7b-chat,0.000054,0.829932,1176


In [29]:
# Convert samples to a long format for later evaluation
def prepare_samples(data, model_names):
    parts = []

    for model in model_names:
        part = data[
            ["sample_id", "eval_name", "category"]
        ].copy()

        part["model"] = model
        part["score"] = pd.to_numeric(
            data[model], errors="raise"
        )
        part["cost"] = pd.to_numeric(
            data[f"{model}|total_cost"], errors="raise"
        )

        parts.append(part)

    return pd.concat(parts, ignore_index=True)


estimation_samples = prepare_samples(estimation_df, models)
evaluation_samples = prepare_samples(evaluation_df, models)

print("Estimation rows:", len(estimation_samples))
print("Evaluation rows:", len(evaluation_samples))

Estimation rows: 24134
Evaluation rows: 6039


In [ ]:
import json

# Save prepared datasets and estimated parameters
output_dir = Path("data") / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

#File with the hit rate, average cost and numer of samples
all_parameters.to_csv(
    output_dir / "all_model_parameters.csv", index=False
)

#File with the same parameters but only of the three selected models
model_parameters.to_csv(
    output_dir / "model_parameters.csv", index=False
)

#File with the result and cost o feach question and model of the 80%
#used to calculate the parameters
estimation_samples.to_csv(
    output_dir / "estimation_samples.csv", index=False
)

#File with the result and cost of the 20% reserved to evaluate 
#after the asignation
evaluation_samples.to_csv(
    output_dir / "evaluation_samples.csv", index=False
)

#Number of questions of each category in both groups (train+test)
split_summary.to_csv(output_dir / "split_summary.csv")

#Indicates which questions belong to train and which to test
split_assignments = pd.concat(
    [
        estimation_df[
            ["sample_id", "eval_name", "category"]
        ].assign(split="estimation"),
        evaluation_df[
            ["sample_id", "eval_name", "category"]
        ].assign(split="evaluation")
    ],
    ignore_index=True
)

split_assignments.to_csv(
    output_dir / "split_assignments.csv", index=False
)

#Document preparation choices
metadata = {
    "source": "https://huggingface.co/datasets/withmartian/routerbench",
    "source_file": data_file.name,
    "random_seed": random_seed,
    "estimation_fraction_target": 0.8,
    "split_method": "Random split within each benchmark",
    "category_mapping": category_mapping,
    "selected_models": selected_models,
    "duplicate_rule": "Keep first exact prompt after stripping whitespace",
    "duplicates_removed": int(duplicate_count),
    "quality_measure": "Mean binary score",
    "category_weighting": "Equal weight per estimation sample",
    "cost_measure": "Mean recorded benchmark cost per query",
    "cost_currency": "Pending verification from source documentation"
}

with open(
    output_dir / "preparation_metadata.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(metadata, file, indent=4, ensure_ascii=False)

print("Files saved to:", output_dir.resolve())

Files saved to: C:\Users\tcont\Documents\3iMAT\Optimización\FinalProject\OptimizationFinalProject\data\processed
